# Chapter 4 · Tools and Interoperability
**Mastering Agentic AI** — Manning Publications

## What this notebook covers
- How tool calling works internally (3 stages)
- Five principles for production-ready tools
- Building a resilient get_nutrition tool
- MCP: Host, Client, Server architecture
- Minimal MCP client and server implementation
- Composio and tooling ecosystems
- Agent Skills + Tools working together



## 4.0 · Setup

In [ ]:
# !pip install anthropic mcp requests python-dotenv composio-crewai
import os, json, logging, subprocess
from anthropic import Anthropic
from dotenv import load_dotenv
load_dotenv()
assert os.getenv('ANTHROPIC_API_KEY'), 'Set ANTHROPIC_API_KEY — see Appendix A.'
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
client = Anthropic()
MODEL = 'claude-opus-4-5'
print('Ready')

## 4.1 · Tool Calling Internals

Three stages:
1. **Configuration** — inject tool manifest into context (name + schema + docstring)
2. **Invocation** — model emits structured JSON intent
3. **Execution** — YOUR code runs the function; result returned to model

The model never executes code. It describes intent. You decide whether to run it.

In [ ]:
# Stage 2 example: what the model emits
INVOCATION_EXAMPLE = {
    'tool_name': 'get_nutrition',
    'arguments': {'food_item': '1 banana'}
}
print('Model emits INTENT:')
print(json.dumps(INVOCATION_EXAMPLE, indent=2))
print()
print('Key insight: the model NEVER calls the function.')
print('YOUR orchestrator calls the function and returns the result.')

## 4.2 · Five Principles for Production-Ready Tools

In [ ]:
import requests

NUTRITION_API_URL = 'https://trackapi.nutritionix.com/v2/natural/nutrients'
API_KEY = os.getenv('NUTRITIONIX_API_KEY', 'DEMO')
APP_ID  = os.getenv('NUTRITIONIX_APP_ID',  'DEMO')

def get_nutrition(food_item: str) -> str:
    # Principle 1: Docstring is the LLM's user interface
    '''
    Fetches nutritional information for a single food item from NutritionIX.
    For raw ingredients only (e.g., '1 banana', '150g chicken breast').
    Not suitable for complex meals or restaurant dishes.
    Returns: JSON with status=success|not_found|error and nutritional data.
    '''
    logging.info(f"Tool called: food_item={food_item!r}")  # Principle 5: Log

    if not food_item or not isinstance(food_item, str):    # Principle 3: Validate
        return json.dumps({'status': 'error', 'message': 'food_item must be non-empty string'})

    headers = {'x-app-id': APP_ID, 'x-app-key': API_KEY, 'Content-Type': 'application/json'}
    body    = {'query': food_item.lower().strip()}

    try:                                                   # Principle 4: Build for failure
        r = requests.post(NUTRITION_API_URL, headers=headers, json=body, timeout=10)
        r.raise_for_status()
        data = r.json()
        if not data.get('foods'):
            return json.dumps({'status': 'not_found', 'message': f'{food_item!r} not found'})
        f = data['foods'][0]
        return json.dumps({'status': 'success', 'data': {
            'item': f.get('food_name'), 'calories': f.get('nf_calories'),
            'protein_g': f.get('nf_protein'), 'carbs_g': f.get('nf_total_carbohydrate')
        }})
    except requests.exceptions.Timeout:
        return json.dumps({'status': 'error', 'message': 'API timed out'})
    except Exception as e:
        logging.critical(f'Unexpected: {e}', exc_info=True)
        return json.dumps({'status': 'error', 'message': 'Unexpected internal error'})

# Mock for demos without API keys
NUTRITION_DB = {'apple': {'calories':95,'protein_g':0.5,'carbs_g':25},
                'oats': {'calories':150,'protein_g':5.0,'carbs_g':27},
                'salmon': {'calories':208,'protein_g':28.0,'carbs_g':0},
                'banana': {'calories':89,'protein_g':1.1,'carbs_g':23},
                'chicken breast': {'calories':165,'protein_g':31.0,'carbs_g':0}}

def get_nutrition_mock(food_item: str) -> str:
    key = food_item.strip().lower()
    data = NUTRITION_DB.get(key)
    if data:
        return json.dumps({'status': 'success', 'data': {'item': key, **data}})
    return json.dumps({'status': 'not_found', 'message': f'{food_item!r} not in mock DB'})

print(get_nutrition_mock('banana'))
print(get_nutrition_mock('pizza'))  # structured not_found — not a crash

## 4.3 · MCP Architecture

In [ ]:
print('''
MCP — Model Context Protocol (Anthropic, 2024)

Three components:
  Host   — the AI application; manages tool discovery and routing
  Client — one per server; translates host requests to protocol calls
  Server — standalone program exposing tools via JSON-RPC 2.0

Flow: LLM → intent → Host → Client → Server → executes → result → LLM

Before MCP: framework-specific integrations, rebuilt for every ecosystem
After MCP:  write once, use everywhere (CrewAI, LangChain, Cursor, Claude Desktop)

Think of MCP as USB-C for agent tools.
''')

## 4.4 · Minimal MCP Client

In [ ]:
class SimpleMCPClient:
    '''Minimal MCP client (stdio transport). For production: pip install mcp'''
    def __init__(self, server_command: list):
        self.proc = subprocess.Popen(
            server_command,
            stdin=subprocess.PIPE, stdout=subprocess.PIPE, text=True)
        self._id = 0

    def _rpc(self, method, params=None):
        self._id += 1
        req = json.dumps({'jsonrpc':'2.0','id':self._id,
                          'method':method,'params':params or {}})
        self.proc.stdin.write(req + chr(10))
        self.proc.stdin.flush()
        return json.loads(self.proc.stdout.readline()).get('result', {})

    def list_tools(self):   return self._rpc('tools/list').get('tools', [])
    def call_tool(self, name, arguments): return self._rpc('tools/call', {'name':name,'arguments':arguments})
    def close(self): self.proc.terminate()

print('SimpleMCPClient ready.')
print('Usage:')
print('  client = SimpleMCPClient(["python", "diet_coach_mcp_server.py"])')
print('  tools  = client.list_tools()')
print('  result = client.call_tool("lookup_nutrition", {"food_item": "oats"})')

## 4.5 · MCP Server (diet_coach_mcp_server.py)

In [ ]:
# Writing the MCP server to disk
mcp_server_code = open("diet_coach_mcp_server.py").read() if False else """
See diet_coach_mcp_server.py in this chapter directory.
Install: pip install mcp
Run:     python diet_coach_mcp_server.py
"""

import os
server_path = "diet_coach_mcp_server.py"
# The file is already written to this directory
# To use: client = SimpleMCPClient(["python", server_path])
print(f"MCP server file: {server_path}")
print("See diet_coach_mcp_server.py in this chapter directory for the full server code.")


## 4.6 · Composio — Tooling Ecosystem Pattern

In [ ]:
# Composio usage pattern — requires: pip install composio-crewai
# and COMPOSIO_API_KEY in your .env file (see Appendix A)

print('''
Composio provides 200+ pre-built connectors (GitHub, Slack, Google Drive, ...).

Core benefit: instead of writing API integration code, you get tools that are
already LLM-ready — with correct schemas, auth handling, and docstrings.

Pattern:
  github_tools = composio.tools.get(user_id=uid, toolkits=["GITHUB"])
  agent = Agent(role="...", tools=github_tools, ...)
  # Agent selects the right tool autonomously based on user intent

When to use Composio:  common APIs (GitHub, Slack, Calendar)
When to use custom:    domain-specific logic (our get_nutrition)
Best practice:         hybrid — both in the same agent system
''')

## 4.7 · Agent Skills + Tools: The Full Pattern

In [ ]:
SKILL_CONTENT = '''# Nutrition Assessment Skill
Step 1 — Establish Baseline: eating pattern, restrictions, goals, activity.
Step 2 — Identify Deficits: protein (<0.8g/kg), fibre (<25g/day), micronutrients.
Step 3 — Prioritise: high-impact, low-effort first.
Step 4 — One Measurable Goal: concrete and time-bound. One goal only.'''

TOOLS_FOR_AGENT = [
    {'name':'lookup_nutrition',
     'description':'Return macro-nutrient data for a food item per 100g.',
     'input_schema':{'type':'object','properties':{'food_item':{'type':'string'}},'required':['food_item']}},
    {'name':'suggest_meal',
     'description':'Suggest a meal matching a dietary goal (e.g., high protein, low carb).',
     'input_schema':{'type':'object','properties':{'goal':{'type':'string'},'max_calories':{'type':'integer'}},'required':['goal']}},
]

TOOL_FNS = {
    'lookup_nutrition': get_nutrition_mock,
    'suggest_meal': lambda goal, max_calories=600: json.dumps({'suggestions': ['oats','salmon','chicken breast']}),
}

def run_skill_guided_agent(user_message):
    system = (f'You are an AI Diet Coach.
[Skill]
{SKILL_CONTENT}
'
              'Use tools for data. Apply skill protocol for full assessments.')
    messages = [{'role':'user','content':user_message}]

    while True:
        response = client.messages.create(
            model=MODEL, max_tokens=1024, system=system,
            tools=TOOLS_FOR_AGENT, messages=messages)
        messages.append({'role':'assistant','content':response.content})

        if response.stop_reason == 'end_turn':
            return ''.join(b.text for b in response.content if b.type == 'text')

        tool_results = []
        for block in response.content:
            if block.type == 'tool_use':
                fn = TOOL_FNS.get(block.name)
                result = fn(**block.input) if fn else json.dumps({'error':'unknown'})
                tool_results.append({'type':'tool_result','tool_use_id':block.id,'content':result})
        messages.append({'role':'user','content':tool_results})

answer = run_skill_guided_agent('I want high protein dinner under 500 calories. What should I eat?')
print('Coach:', answer)

## 4.8 · Chapter Summary

| Concept | Key insight |
|---|---|
| Tool calling internals | 3 stages: configure → invoke → execute (you are the executor) |
| Production tool design | 5 principles: docstring, atomic, defensive, failure-ready, logged |
| MCP architecture | Decouples reasoning from execution across frameworks |
| SimpleMCPClient | JSON-RPC 2.0 over stdio — what the protocol actually does |
| MCP server | FastMCP wraps tools for any MCP-compatible host |
| Composio | Tooling ecosystem: toolkits over individual functions |
| Skill + Tool | Skills = HOW to think; Tools = WHAT to say |

**Next:** Chapter 5 — Memory: agents that remember across sessions.

---
*Mastering Agentic AI · Manning Publications*